# 39 — Estimation and bias–variance

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/039-estimation-bias-variance.ipynb)

Expected squared error splits into bias squared plus variance plus noise. This notebook simulates one estimator many times, measures both terms, and plots the U-shaped total as a shrinkage dial moves.

In [ ]:
import numpy as np  # arrays and simulation
rng = np.random.default_rng(1)  # reproducible draws
mu, sigma, n = 10.0, 2.0, 4  # true mean, spread, sample size
draws = rng.normal(mu, sigma, size=(50000, n)).mean(axis=1)  # sample means
bias = draws.mean() - mu  # systematic miss
var = draws.var()  # wobble around the mean
print("bias", bias)  # about 0
print("variance", var)  # about 1.0
print("MSE", ((draws - mu) ** 2).mean())  # about 1.0

## The three terms

The sample mean is unbiased: its average over 50,000 simulated studies is essentially `mu`, so the bias term is 0. Its variance is `sigma^2 / n = 4 / 4 = 1`, and with no noise term the MSE is 1.0.

### Buying variance with bias

Shrink the estimate toward a fixed value: `theta_hat = w * x_bar + (1 - w) * 9`. At `w = 1` this is the sample mean; at `w = 0.95` it is pulled slightly toward 9. The bias grows and the variance shrinks, and the total can fall.

In [ ]:
import matplotlib.pyplot as plt  # plotting
ws = np.linspace(0.7, 1.0, 61)  # shrink weights from 0.7 to 1.0
rows = []  # collect one row per weight
for w in ws:  # sweep the dial
    est = w * draws + (1 - w) * 9.0  # shrink toward 9
    rows.append([(est.mean() - mu) ** 2, est.var()])  # bias^2, variance
rows = np.array(rows)  # shape (61, 2)
total = rows.sum(axis=1)  # bias^2 + variance
plt.figure(figsize=(7, 3))  # wide figure
plt.plot(ws, rows[:, 0], label="bias squared")  # falling term
plt.plot(ws, rows[:, 1], label="variance")  # rising term
plt.plot(ws, total, label="total", linewidth=2.5)  # the U
plt.xlabel("weight on the sample mean")  # horizontal label
plt.ylabel("error")  # vertical label
plt.legend()  # name the curves
plt.title("a little shrinkage lowers total error")  # takeaway
plt.show()  # draw it
best = ws[total.argmin()]  # weight at the bottom
print("best weight", best)  # slightly below 1.0

## Reading the U

`bias squared` falls as the weight approaches 1, because the estimator stops being pulled toward 9. `variance` rises over the same range. The total is their sum, and its minimum sits just below 1.0 — a small bias is worth paying for.

### The same shape in model size

Replace the shrinkage weight with polynomial degree and the curves have the same meaning. A rigid model sits on the left with high bias, a flexible one on the right with high variance, and validation error picks the bottom. Training error alone always prefers the right.

In [ ]:
def mse_parts(bias, variance, noise=0.0):  # decompose expected error
    return float(bias ** 2), float(variance), float(noise)  # three terms
b, v, e = mse_parts(-0.3, 0.9025, 0.0)  # the lesson's numbers
print("bias^2", b)  # 0.09
print("variance", v)  # 0.9025
print("total", b + v + e)  # 0.9925
print("beats unbiased?", b + v + e < 1.0)  # True